In [ ]:
### DATA LOCATION ###
# Set DATA_ROOT to the directory containing the behavior MATLAB (.mat) event-log files.
# Either edit the default below or set the PANICCIA_DATA_ROOT environment variable.
# The data are not included in this repository (see data/README.md).
import os
DATA_ROOT = os.environ.get('PANICCIA_DATA_ROOT', './data')


In [ ]:
###updating delta lick rate code to take into account every trial

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
import seaborn as sns
import os
import subprocess
import time
import pandas as pd
import pickle
import math
import scipy.stats as stats
import scipy.io as sio
from scipy import interpolate
from sklearn.metrics import roc_auc_score as auROC
import matplotlib.cm as cm
import matplotlib.colors as colors
import matplotlib.colorbar as colorbar
import matplotlib.patches as mpatches
import seaborn as sns
import sys
import re
import random
from scipy.stats import ttest_ind
from IPython.core.display import display, HTML
display(HTML("<style>.container { width:100% !important; }</style>"))

import numpy as np
from pathlib import Path  # Added this import

In [ ]:
###this portion of the script will split up the responses into SDT-like metrics and visualize them as a pie chart 
####THIS IS DONE FOR THE CUE PERIOD SPECIFICALLY USING THE DELTA LICK RATE


import os
import numpy as np
import pandas as pd
import scipy.io as sio
import matplotlib.pyplot as plt
from pathlib import Path

def calculate_lick_rates_and_counts(cues, licks, window):
    """Calculate licking metrics for a set of cues."""
    # Calculate baseline licks (3 second before cue)
    baseline_licks = np.array([np.sum((licks > t - 3000) & (licks < t)) for t in cues])
    
    # Calculate response licks during cue period (3 seconds)
    cue_period_licks = np.array([np.sum((licks > t) & (licks < t + 3000)) for t in cues])
    
    # Calculate reward period licks (3 seconds after cue)
    reward_period_licks = np.array([np.sum((licks > t + 3000) & (licks < t + 6000)) for t in cues])
    
    # Calculate rates
    baseline_rate = baseline_licks / 3.0  # per second
    cue_period_rate = cue_period_licks / 3.0  # per second
    reward_period_rate = reward_period_licks / 3.0  # per second
    
    # Calculate delta rates
    cue_delta = cue_period_rate - baseline_rate
    reward_delta = reward_period_rate - baseline_rate
    
    return cue_delta, reward_delta, cue_period_licks, reward_period_licks

def is_trial_rewarded(trial_time, fxdpumps, reward_delay=3000, tolerance=100):
    """
    Check if a trial was rewarded by looking for reward delivery 3 seconds after cue.
    """
    if fxdpumps.size == 0:
        return False
    
    expected_reward_time = trial_time + reward_delay
    reward_delivered = np.any(
        (fxdpumps >= expected_reward_time - tolerance) & 
        (fxdpumps <= expected_reward_time + tolerance)
    )
    
    return reward_delivered

def analyze_matlab_file(filepath):
    """Analyze a single MATLAB file and return processed data."""
    # Load MATLAB file
    behaviordata = sio.loadmat(filepath)
    
    # Extract data
    eventlog = np.squeeze(behaviordata['eventlog'])
    licks = np.squeeze(behaviordata['licks'])
    fxdpumps = np.squeeze(behaviordata['fxdpumps'])
    
    # Extract cue timestamps
    cueplus = eventlog[eventlog[:,0]==5,1]  # CS+
    cueminus = eventlog[eventlog[:,0]==6,1]  # CS-
    
    # Calculate metrics
    cueplus_cue_delta, cueplus_reward_delta, cueplus_raw_cue, cueplus_raw_reward = calculate_lick_rates_and_counts(
        cueplus, licks, 3000)
    cueminus_cue_delta, cueminus_reward_delta, cueminus_raw_cue, cueminus_raw_reward = calculate_lick_rates_and_counts(
        cueminus, licks, 3000)
    
    # For each CS+ trial, determine if it was rewarded
    cueplus_rewarded = [is_trial_rewarded(t, fxdpumps) for t in cueplus]
    
    return {
        'cueplus_cue_delta': cueplus_cue_delta,
        'cueplus_reward_delta': cueplus_reward_delta,
        'cueminus_cue_delta': cueminus_cue_delta,
        'cueminus_reward_delta': cueminus_reward_delta,
        'cueplus_raw_cue': cueplus_raw_cue,
        'cueminus_raw_cue': cueminus_raw_cue,
        'cueplus_raw_reward': cueplus_raw_reward,
        'cueminus_raw_reward': cueminus_raw_reward,
        'cueplus_rewarded': cueplus_rewarded,  # Add reward status
        'fxdpumps': fxdpumps,
        'cueplus': cueplus,
        'cueminus': cueminus
    }

def classify_response(phase, cue_type, is_high_licking):
    """
    Classify response type based on learning phase, cue type, and licking behavior.
    
    Parameters:
    phase: str - Learning phase (e.g., 'Early', 'Late', 'Suc-Omit', 'Rev-Early', 'Rev-Late')
    cue_type: str - 'CS+' or 'CS-'
    is_high_licking: bool - Whether trial had high licking rate
    """
    # Check if we're in a reversal phase
    is_reversal = phase.startswith('Rev-')
    
    if not is_reversal:
        # Initial learning phases
        if cue_type == 'CS+':
            return 'correct response' if is_high_licking else 'incorrect withholding'
        else:  # CS-
            return 'incorrect response' if is_high_licking else 'correct withholding'
    else:
        # Reversal phases
        if cue_type == 'CS+':
            return 'incorrect response' if is_high_licking else 'correct withholding'
        else:  # CS-
            return 'correct response' if is_high_licking else 'incorrect withholding'

def build_session_data_dictionary(basedir):
    """Build hierarchical data structure for session-by-session analysis."""
    data_dict = {
        'Early': {},
        'Late': {},
        'SucOmit': {},
        'CueOmit': {},
        'Rev-Early': {},
        'Rev-Late': {}
    }
    
    THRESHOLD = 1.5  # Hz threshold for high/low licking classification
    
    # Define all possible phase directories to check
    phases = {
        'Early': ['Early'],
        'Late': ['Late'],
        'SucOmit': ['SucOmit'],
        'CueOmit': ['CueOmit'],
        'Rev-Early': ['Rev_Early'],
        'Rev-Late': ['Rev_Late']  
    }
    
    for phase_name, phase_variants in phases.items():
        print(f"\nProcessing {phase_name} phase...")
        
        # Try each variant of the phase name
        phase_found = False
        for variant in phase_variants:
            phase_dir = Path(basedir) / variant
            if not phase_dir.exists():
                continue
                
            phase_found = True
            print(f"Found directory: {variant}")
            
            # Process all .mat files in this phase
            for matfile in phase_dir.glob('*.mat'):
                print(f"Processing file: {matfile.name}")
                
                try:
                    # Process MATLAB file
                    metrics = analyze_matlab_file(matfile)
                    session_id = matfile.stem
                    
                    # Initialize session data structure
                    data_dict[phase_name][session_id] = {
                        'CS+': {
                            'trials': [],
                            'responses': [],
                            'cue_delta_rates': metrics['cueplus_cue_delta'].tolist(),
                            'reward_delta_rates': metrics['cueplus_reward_delta'].tolist(),
                            'cue_timestamps': metrics['cueplus'].tolist()
                        },
                        'CS-': {
                            'trials': [],
                            'responses': [],
                            'cue_delta_rates': metrics['cueminus_cue_delta'].tolist(),
                            'reward_delta_rates': metrics['cueminus_reward_delta'].tolist(),
                            'cue_timestamps': metrics['cueminus'].tolist()
                        }
                    }
                    
                    # Add special tracking for SucOmit CS+ trials
                    if phase_name == 'SucOmit':
                        data_dict[phase_name][session_id]['CS+'].update({
                            'rewarded_trials': [],
                            'unrewarded_trials': [],
                            'rewarded_responses': [],
                            'unrewarded_responses': [],
                            'rewarded_indices': [],
                            'unrewarded_indices': []
                        })
                    
                    # Process CS+ trials
                    for i, delta_rate in enumerate(metrics['cueplus_cue_delta']):
                        is_high = delta_rate > THRESHOLD
                        response = classify_response(phase_name, 'CS+', is_high)
                        trial_data = {
                            'cue_delta_rate': delta_rate,
                            'reward_delta_rate': metrics['cueplus_reward_delta'][i],
                            'high_low': int(is_high),
                            'response': response
                        }
                        
                        data_dict[phase_name][session_id]['CS+']['trials'].append(trial_data)
                        data_dict[phase_name][session_id]['CS+']['responses'].append(response)
                        
                        # Special handling for SucOmit phase
                        if phase_name == 'SucOmit':
                            if metrics['cueplus_rewarded'][i]:
                                data_dict[phase_name][session_id]['CS+']['rewarded_trials'].append(trial_data)
                                data_dict[phase_name][session_id]['CS+']['rewarded_responses'].append(response)
                                data_dict[phase_name][session_id]['CS+']['rewarded_indices'].append(i)
                            else:
                                data_dict[phase_name][session_id]['CS+']['unrewarded_trials'].append(trial_data)
                                data_dict[phase_name][session_id]['CS+']['unrewarded_responses'].append(response)
                                data_dict[phase_name][session_id]['CS+']['unrewarded_indices'].append(i)
                    
                    # Process CS- trials
                    for i, delta_rate in enumerate(metrics['cueminus_cue_delta']):
                        is_high = delta_rate > THRESHOLD
                        response = classify_response(phase_name, 'CS-', is_high)
                        trial_data = {
                            'cue_delta_rate': delta_rate,
                            'reward_delta_rate': metrics['cueminus_reward_delta'][i],
                            'high_low': int(is_high),
                            'response': response
                        }
                        data_dict[phase_name][session_id]['CS-']['trials'].append(trial_data)
                        data_dict[phase_name][session_id]['CS-']['responses'].append(response)
                    
                except Exception as e:
                    print(f"Error processing {matfile}: {str(e)}")
                    continue
                
                # Print summary statistics for verification
                print(f"\nSummary for session {session_id}:")
                print_session_summary(data_dict[phase_name][session_id], phase_name)
            
            if phase_found:
                break
    
    return data_dict

def print_session_summary(session_data, phase_name):
    """Print summary statistics for a session."""
    # Print regular summary for both cue types
    for cue_type in ['CS+', 'CS-']:
        responses = session_data[cue_type]['responses']
        if responses:
            unique, counts = np.unique(responses, return_counts=True)
            proportions = dict(zip(unique, counts / len(responses) * 100))
            print(f"\n{cue_type}:")
            for resp_type, prop in proportions.items():
                print(f"{resp_type}: {prop:.1f}%")
    
    # Print additional summary for Suc-Omit CS+ trials
    if phase_name == 'SucOmit':
        print("\nCS+ Rewarded trials:")
        rewarded_resp = session_data['CS+']['rewarded_responses']
        if rewarded_resp:
            unique, counts = np.unique(rewarded_resp, return_counts=True)
            proportions = dict(zip(unique, counts / len(rewarded_resp) * 100))
            for resp_type, prop in proportions.items():
                print(f"{resp_type}: {prop:.1f}%")
        
        print("\nCS+ Unrewarded trials:")
        unrewarded_resp = session_data['CS+']['unrewarded_responses']
        if unrewarded_resp:
            unique, counts = np.unique(unrewarded_resp, return_counts=True)
            proportions = dict(zip(unique, counts / len(unrewarded_resp) * 100))
            for resp_type, prop in proportions.items():
                print(f"{resp_type}: {prop:.1f}%")

def visualize_phase_responses(data_dict, results_dir):
    """Create pie charts showing aggregated responses for each phase."""
    colors = {
        'correct response': 'greenyellow',
        'correct withholding': 'greenyellow',
        'incorrect response': 'darkslategrey',
        'incorrect withholding': 'grey'
    }
    
    # Create directory for phase plots
    phase_plots_dir = Path(results_dir) / 'phase_plots'
    phase_plots_dir.mkdir(exist_ok=True)
    
    # Process each phase
    for phase_name, phase_data in data_dict.items():
        print(f"\nProcessing visualizations for {phase_name}")
        
        # Collect all responses across animals
        all_responses = {
            'CS+': [],
            'CS-': []
        }
        
        # For Suc-Omit, also track reward status
        if phase_name == 'SucOmit':
            all_responses['CS+_rewarded'] = []
            all_responses['CS+_unrewarded'] = []
        
        # Aggregate responses across all animals
        for animal_data in phase_data.values():
            all_responses['CS+'].extend(animal_data['CS+']['responses'])
            all_responses['CS-'].extend(animal_data['CS-']['responses'])
            
            if phase_name == 'SucOmit':
                all_responses['CS+_rewarded'].extend(animal_data['CS+']['rewarded_responses'])
                all_responses['CS+_unrewarded'].extend(animal_data['CS+']['unrewarded_responses'])
        
        # Create overall phase response plot
        fig, ax = plt.subplots(figsize=(12, 10))
        fig.suptitle(f'{phase_name} Phase - All Animals\nTotal Responses', fontsize=16)
        
        plot_data = []
        
        # Process CS+ responses (50% of pie)
        cs_plus_responses = all_responses['CS+']
        if cs_plus_responses:
            unique, counts = np.unique(cs_plus_responses, return_counts=True)
            props = dict(zip(unique, counts / len(cs_plus_responses) * 50))  # Scale to 50% of pie
            for resp_type, prop in props.items():
                plot_data.append({
                    'label': f'CS+ {resp_type}\n({counts[list(unique).index(resp_type)]} trials)',
                    'size': prop,
                    'color': colors[resp_type]
                })
        
        # Process CS- responses (50% of pie)
        cs_minus_responses = all_responses['CS-']
        if cs_minus_responses:
            unique, counts = np.unique(cs_minus_responses, return_counts=True)
            props = dict(zip(unique, counts / len(cs_minus_responses) * 50))  # Scale to 50% of pie
            for resp_type, prop in props.items():
                plot_data.append({
                    'label': f'CS- {resp_type}\n({counts[list(unique).index(resp_type)]} trials)',
                    'size': prop,
                    'color': colors[resp_type]
                })
        
        if plot_data:
            sizes = [d['size'] for d in plot_data]
            labels = [d['label'] for d in plot_data]
            colors_list = [d['color'] for d in plot_data]
            
            wedges, texts, autotexts = ax.pie(
                sizes, 
                labels=labels, 
                colors=colors_list,
                autopct='%1.1f%%', 
                startangle=90
            )
            plt.setp(autotexts, weight='bold')
            plt.setp(texts, size=8)
            
            # Add total trial counts to title
            total_csplus = len(cs_plus_responses)
            total_csminus = len(cs_minus_responses)
            plt.title(f'Total Trials: CS+ = {total_csplus}, CS- = {total_csminus}', 
                     pad=30, fontsize=12)
        
        save_path = phase_plots_dir / f'{phase_name}_overall_responses_03-18-25_all-early.pdf'
        plt.savefig(save_path, bbox_inches='tight', dpi=300)
        plt.show()
        
        # For Suc-Omit, create additional reward-specific plot
        if phase_name == 'SucOmit':
            fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(20, 10))
            fig.suptitle('Suc-Omit Phase - CS+ Responses by Reward Status', fontsize=16)
            
            # Plot rewarded CS+ trials
            rewarded_responses = all_responses['CS+_rewarded']
            if rewarded_responses:
                unique, counts = np.unique(rewarded_responses, return_counts=True)
                props = dict(zip(unique, counts / len(rewarded_responses) * 100))
                
                labels = [f'{resp_type}\n({counts[list(unique).index(resp_type)]} trials)' 
                         for resp_type in props.keys()]
                sizes = list(props.values())
                colors_list = [colors[resp_type] for resp_type in props.keys()]
                
                wedges, texts, autotexts = ax1.pie(
                    sizes,
                    labels=labels,
                    colors=colors_list,
                    autopct='%1.1f%%',
                    startangle=90
                )
                plt.setp(autotexts, weight='bold')
                plt.setp(texts, size=8)
                ax1.set_title(f'Rewarded CS+ Trials (n={len(rewarded_responses)})')
            
            # Plot unrewarded CS+ trials
            unrewarded_responses = all_responses['CS+_unrewarded']
            if unrewarded_responses:
                unique, counts = np.unique(unrewarded_responses, return_counts=True)
                props = dict(zip(unique, counts / len(unrewarded_responses) * 100))
                
                labels = [f'{resp_type}\n({counts[list(unique).index(resp_type)]} trials)' 
                         for resp_type in props.keys()]
                sizes = list(props.values())
                colors_list = [colors[resp_type] for resp_type in props.keys()]
                
                wedges, texts, autotexts = ax2.pie(
                    sizes,
                    labels=labels,
                    colors=colors_list,
                    autopct='%1.1f%%',
                    startangle=90
                )
                plt.setp(autotexts, weight='bold')
                plt.setp(texts, size=8)
                ax2.set_title(f'Unrewarded CS+ Trials (n={len(unrewarded_responses)})')
            
            save_path = phase_plots_dir / f'{phase_name}_reward_specific_responses_1-5hz_03-18-2025_all-Early.pdf'
            plt.savefig(save_path, bbox_inches='tight', dpi=300)
            plt.show()
            
        # Print summary statistics
        print(f"\nSummary for {phase_name} phase:")
        for cue_type in ['CS+', 'CS-']:
            responses = all_responses[cue_type]
            if responses:
                unique, counts = np.unique(responses, return_counts=True)
                proportions = dict(zip(unique, counts / len(responses) * 100))
                print(f"\n{cue_type} (Total trials: {len(responses)}):")
                for resp_type, prop in proportions.items():
                    print(f"{resp_type}: {prop:.1f}% ({counts[list(unique).index(resp_type)]} trials)")
        
        if phase_name == 'SucOmit':
            print("\nCS+ Reward-specific breakdown:")
            print(f"Rewarded trials: {len(all_responses['CS+_rewarded'])}")
            print(f"Unrewarded trials: {len(all_responses['CS+_unrewarded'])}")

# Define base directory
basedir = DATA_ROOT
results_dir = Path(basedir) / 'Results'
results_dir.mkdir(exist_ok=True)

# Build data dictionary
print("Processing MATLAB files...")
data_dict = build_session_data_dictionary(basedir)

# Create visualizations
print("\nCreating visualizations...")
visualize_phase_responses(data_dict, results_dir)

In [ ]:
##Chi-Square analyses on the response type data

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from pathlib import Path
import seaborn as sns
import warnings

def perform_chi_square_analysis(data_dict, save_dir=None, combined_analysis=True):
    """
    Perform chi-square analysis on response types across different experimental phases,
    accounting for phase-specific response types.
    
    Parameters:
    -----------
    data_dict : dict
        The hierarchical data dictionary from the original analysis
    save_dir : Path or str, optional
        Directory to save results
    combined_analysis : bool, default=True
        If True, also perform analysis with all response types combined across cue types
    """
    if save_dir:
        save_dir = Path(save_dir)
        stats_dir = save_dir / 'chi_square_stats'
        stats_dir.mkdir(exist_ok=True)
    
    # Define the phase comparisons we want to analyze
    phase_comparisons = [
        ['Early', 'Late'],
        ['Late', 'SucOmit'],
        ['Rev-Early', 'Rev-Late']
    ]
    
    # Define RELEVANT response types for each cue type and phase group
    # For normal phases (Early, Late, SucOmit)
    normal_phase_response_types = {
        'CS+': ['correct response', 'incorrect withholding'],
        'CS-': ['correct withholding', 'incorrect response']
    }
    
    # For reversal phases (Rev-Early, Rev-Late)
    reversal_phase_response_types = {
        'CS+': ['correct withholding', 'incorrect response'],
        'CS-': ['correct response', 'incorrect withholding']
    }
    
    # Collect aggregated response counts for each phase and cue type
    phase_response_counts = {}
    
    for phase_name, phase_data in data_dict.items():
        if not phase_data:  # Skip empty phases
            continue
            
        # Initialize counts for this phase
        phase_response_counts[phase_name] = {
            'CS+': {},
            'CS-': {},
            'Combined': {}
        }
        
        # Determine which response types are relevant for this phase
        is_reversal = phase_name.startswith('Rev-')
        relevant_types = reversal_phase_response_types if is_reversal else normal_phase_response_types
        
        # Initialize response types for this phase
        for cue_type in ['CS+', 'CS-']:
            for resp_type in relevant_types[cue_type]:
                phase_response_counts[phase_name][cue_type][resp_type] = 0
                phase_response_counts[phase_name]['Combined'][resp_type] = 0
        
        # Aggregate responses across all sessions in this phase
        for session_id, session_data in phase_data.items():
            for cue_type in ['CS+', 'CS-']:
                responses = session_data[cue_type]['responses']
                
                for response in responses:
                    if response in relevant_types[cue_type]:
                        phase_response_counts[phase_name][cue_type][response] += 1
                        # Also add to combined counts
                        phase_response_counts[phase_name]['Combined'][response] += 1
    
    # Print summary of collected data
    print("\nSummary of Response Counts by Phase and Cue Type:")
    for phase, cue_data in phase_response_counts.items():
        print(f"\n{phase} Phase:")
        for cue, resp_counts in cue_data.items():
            if cue == 'Combined' and not combined_analysis:
                continue  # Skip combined data if not requested
                
            total = sum(resp_counts.values())
            print(f"  {cue} (Total: {total}):")
            for resp, count in resp_counts.items():
                percentage = (count / total * 100) if total > 0 else 0
                print(f"    {resp}: {count} ({percentage:.1f}%)")
    
    # Perform chi-square tests for each defined comparison
    print("\n\n====== Chi-Square Test Results ======")
    
    # Store results for later visualization
    chi_results = []
    
    for phases in phase_comparisons:
        # Filter out phases that don't exist in our data
        phases = [p for p in phases if p in phase_response_counts]
        
        if len(phases) < 2:
            print(f"Skipping comparison {phases} - insufficient data")
            continue
        
        print(f"\nComparing phases: {' vs '.join(phases)}")
        
        # Determine if these are reversal phases
        is_reversal = all(p.startswith('Rev-') for p in phases)
        
        # Get appropriate response types for these phases
        relevant_types = reversal_phase_response_types if is_reversal else normal_phase_response_types
        
        # Define which cue types to analyze
        cue_types_to_analyze = ['CS+', 'CS-']
        if combined_analysis:
            cue_types_to_analyze.append('Combined')
        
        # Analyze each cue type
        for cue_type in cue_types_to_analyze:
            print(f"\n{cue_type} Analysis:")
            
            # For combined analysis, we need all response types
            if cue_type == 'Combined':
                # Get all possible response types across both cue types
                all_response_types = []
                for ct in ['CS+', 'CS-']:
                    all_response_types.extend(relevant_types[ct])
                # Remove duplicates while preserving order
                response_types_to_use = []
                for rt in all_response_types:
                    if rt not in response_types_to_use:
                        response_types_to_use.append(rt)
            else:
                # For specific cue types, use only relevant response types
                response_types_to_use = relevant_types[cue_type]
            
            # Create contingency table: rows=phases, columns=response types
            contingency_data = []
            for phase in phases:
                if cue_type == 'Combined':
                    # For combined, we need to ensure all response types are present
                    phase_counts = []
                    for resp in response_types_to_use:
                        count = phase_response_counts[phase][cue_type].get(resp, 0)
                        phase_counts.append(count)
                else:
                    # For specific cue types, use only relevant response types
                    phase_counts = [phase_response_counts[phase][cue_type][resp] for resp in response_types_to_use]
                
                contingency_data.append(phase_counts)
            
            contingency_table = np.array(contingency_data)
            
            # Display contingency table
            print(f"  Contingency table (rows=phases, columns=response types):")
            table_df = pd.DataFrame(contingency_data, 
                                   index=phases, 
                                   columns=response_types_to_use)
            print(table_df)
            
            # Verify we have enough data for chi-square
            if contingency_table.shape[1] < 2:
                print("  Error: Need at least 2 response types for chi-square test")
                continue
                
            if np.any(contingency_table.sum(axis=0) == 0):
                print("  Error: One or more response types have zero counts across all phases")
                continue
            
            # Run chi-square test
            try:
                chi2, p, dof, expected = stats.chi2_contingency(contingency_table)
                
                # Check if expected frequencies are adequate
                min_expected = np.min(expected)
                if min_expected < 5:
                    warnings.warn(f"Some expected frequencies are below 5 (min={min_expected:.2f}), chi-square may not be reliable")
                    print(f"  Warning: Some expected frequencies are below 5 (min={min_expected:.2f}), chi-square may not be reliable")
            
            except Exception as e:
                print(f"  Error performing chi-square test: {str(e)}")
                continue
            
            # Display test results
            print(f"  Chi-square value: {chi2:.2f}")
            print(f"  Degrees of freedom: {dof}")
            print(f"  p-value: {p:.4f}")
            print(f"  Significant: {'Yes' if p < 0.05 else 'No'}")
            
            # If significant and we have more than 2 phases, perform post-hoc analysis
            if p < 0.05 and len(phases) > 2:
                print("  Post-hoc analysis (standardized residuals):")
                row_totals = contingency_table.sum(axis=1)
                col_totals = contingency_table.sum(axis=0)
                grand_total = contingency_table.sum()
                
                # Calculate expected values
                expected_table = np.outer(row_totals, col_totals) / grand_total
                
                # Calculate standardized residuals
                standardized_residuals = (contingency_table - expected_table) / np.sqrt(expected_table)
                
                # Create DataFrame for nice output
                resid_df = pd.DataFrame(standardized_residuals, 
                                      index=phases, 
                                      columns=response_types_to_use)
                print(resid_df)
                print("  Note: Absolute values > 1.96 indicate significant contribution (p < 0.05)")
                print("        Absolute values > 2.58 indicate significant contribution (p < 0.01)")
                print("        Positive values indicate more than expected")
                print("        Negative values indicate less than expected")
            
            # Save results for visualization
            chi_results.append({
                'comparison': ' vs '.join(phases),
                'cue_type': cue_type,
                'chi2': chi2,
                'p': p,
                'dof': dof,
                'contingency_table': table_df,
                'phases': phases,
                'response_types': response_types_to_use,
                'is_reversal': is_reversal
            })
    
    
    print("\nChi-square analysis complete. Results saved to:", stats_dir if save_dir else "Not saved")
    return chi_results

# Example usage
if __name__ == "__main__":
    # Define results directory - should match the one used in the original script
    basedir = DATA_ROOT
    results_dir = Path(basedir) / 'Results'
    
    # Perform chi-square analysis with phase-specific response types
    chi_results = perform_chi_square_analysis(data_dict, results_dir, combined_analysis=False)

In [ ]:
##high low licking pie charts (basically a double check for the response types)
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from pathlib import Path
import seaborn as sns
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from pathlib import Path
import seaborn as sns
import warnings

def visualize_licking_response(data_dict, results_dir):
    """
    Create visualizations of high vs. low licking responses for each phase.
    
    Parameters:
    -----------
    data_dict : dict
        The hierarchical data dictionary from the original analysis
    results_dir : Path or str
        Directory to save results
    """
    THRESHOLD = 1.5  # Hz threshold for high/low licking classification (same as original)
    
    # Create directory for licking response plots
    licking_plots_dir = Path(results_dir) / 'licking_response_plots'
    licking_plots_dir.mkdir(exist_ok=True)
    
    # Colors for visualization (as requested)
    colors = {
        'CS+_high': 'darkviolet',
        'CS+_low': 'plum',
        'CS-_high': 'dimgrey',
        'CS-_low': 'lightgrey'
    }
    
    # Process each phase
    for phase_name, phase_data in data_dict.items():
        print(f"\nProcessing licking visualizations for {phase_name}")
        
        # Skip if no data
        if not phase_data:
            print(f"No data for {phase_name}, skipping")
            continue
        
        # Collect all licking rates across sessions
        all_licking_data = {
            'CS+': [],
            'CS-': []
        }
        
        # For Suc-Omit, also track reward status
        if phase_name == 'SucOmit':
            all_licking_data['CS+_rewarded'] = []
            all_licking_data['CS+_unrewarded'] = []
        
        # Aggregate licking data across all sessions
        for session_id, session_data in phase_data.items():
            # Extract licking rates for CS+
            all_licking_data['CS+'].extend(session_data['CS+']['cue_delta_rates'])
            
            # Extract licking rates for CS-
            all_licking_data['CS-'].extend(session_data['CS-']['cue_delta_rates'])
            
            # For SucOmit phase, separate rewarded and unrewarded CS+ trials
            if phase_name == 'SucOmit':
                # Get rewarded indices
                rewarded_indices = session_data['CS+'].get('rewarded_indices', [])
                unrewarded_indices = session_data['CS+'].get('unrewarded_indices', [])
                
                # Extract rewarded trial licking rates
                for idx in rewarded_indices:
                    if idx < len(session_data['CS+']['cue_delta_rates']):
                        all_licking_data['CS+_rewarded'].append(session_data['CS+']['cue_delta_rates'][idx])
                
                # Extract unrewarded trial licking rates
                for idx in unrewarded_indices:
                    if idx < len(session_data['CS+']['cue_delta_rates']):
                        all_licking_data['CS+_unrewarded'].append(session_data['CS+']['cue_delta_rates'][idx])
        
        # Create pie chart for overall phase licking responses
        fig, ax = plt.subplots(figsize=(12, 10))
        fig.suptitle(f'{phase_name} Phase - All Animals\nHigh vs. Low Licking Responses', fontsize=16)
        
        plot_data = []
        
        # Process CS+ licking responses (50% of pie)
        cs_plus_data = all_licking_data['CS+']
        if cs_plus_data:
            # Count high and low licking trials
            high_lick_count = sum(rate > THRESHOLD for rate in cs_plus_data)
            low_lick_count = len(cs_plus_data) - high_lick_count
            
            # Calculate proportions (scale to 50% of pie)
            high_prop = (high_lick_count / len(cs_plus_data)) * 50
            low_prop = (low_lick_count / len(cs_plus_data)) * 50
            
            # Add to plot data
            plot_data.append({
                'label': f'CS+ High licking\n({high_lick_count} trials)',
                'size': high_prop,
                'color': colors['CS+_high']
            })
            plot_data.append({
                'label': f'CS+ Low licking\n({low_lick_count} trials)',
                'size': low_prop,
                'color': colors['CS+_low']
            })
        
        # Process CS- licking responses (50% of pie)
        cs_minus_data = all_licking_data['CS-']
        if cs_minus_data:
            # Count high and low licking trials
            high_lick_count = sum(rate > THRESHOLD for rate in cs_minus_data)
            low_lick_count = len(cs_minus_data) - high_lick_count
            
            # Calculate proportions (scale to 50% of pie)
            high_prop = (high_lick_count / len(cs_minus_data)) * 50
            low_prop = (low_lick_count / len(cs_minus_data)) * 50
            
            # Add to plot data
            plot_data.append({
                'label': f'CS- High licking\n({high_lick_count} trials)',
                'size': high_prop,
                'color': colors['CS-_high']
            })
            plot_data.append({
                'label': f'CS- Low licking\n({low_lick_count} trials)',
                'size': low_prop,
                'color': colors['CS-_low']
            })
        
        # Create pie chart
        if plot_data:
            sizes = [d['size'] for d in plot_data]
            labels = [d['label'] for d in plot_data]
            colors_list = [d['color'] for d in plot_data]
            
            wedges, texts, autotexts = ax.pie(
                sizes, 
                labels=labels, 
                colors=colors_list,
                autopct='%1.1f%%', 
                startangle=90
            )
            plt.setp(autotexts, weight='bold')
            plt.setp(texts, size=8)
            
            # Add total trial counts to title
            total_csplus = len(cs_plus_data)
            total_csminus = len(cs_minus_data)
            plt.title(f'Total Trials: CS+ = {total_csplus}, CS- = {total_csminus}', 
                     pad=30, fontsize=12)
            
            # Save figure
            save_path = licking_plots_dir / f'{phase_name}_licking_responses.pdf'
            plt.savefig(save_path, bbox_inches='tight', dpi=300)
            plt.show()
        
        # For Suc-Omit, create additional reward-specific plot
        if phase_name == 'SucOmit':
            fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(20, 10))
            fig.suptitle('Suc-Omit Phase - CS+ Licking Responses by Reward Status', fontsize=16)
            
            # Process rewarded CS+ trials
            rewarded_data = all_licking_data['CS+_rewarded']
            if rewarded_data:
                # Count high and low licking trials
                high_lick_count = sum(rate > THRESHOLD for rate in rewarded_data)
                low_lick_count = len(rewarded_data) - high_lick_count
                
                # Create pie data
                sizes = [high_lick_count, low_lick_count]
                labels = [f'High licking\n({high_lick_count} trials)', 
                          f'Low licking\n({low_lick_count} trials)']
                colors_list = [colors['CS+_high'], colors['CS+_low']]
                
                wedges, texts, autotexts = ax1.pie(
                    sizes,
                    labels=labels,
                    colors=colors_list,
                    autopct='%1.1f%%',
                    startangle=90
                )
                plt.setp(autotexts, weight='bold')
                plt.setp(texts, size=8)
                ax1.set_title(f'Rewarded CS+ Trials (n={len(rewarded_data)})')
            
            # Process unrewarded CS+ trials
            unrewarded_data = all_licking_data['CS+_unrewarded']
            if unrewarded_data:
                # Count high and low licking trials
                high_lick_count = sum(rate > THRESHOLD for rate in unrewarded_data)
                low_lick_count = len(unrewarded_data) - high_lick_count
                
                # Create pie data
                sizes = [high_lick_count, low_lick_count]
                labels = [f'High licking\n({high_lick_count} trials)', 
                          f'Low licking\n({low_lick_count} trials)']
                colors_list = [colors['CS+_high'], colors['CS+_low']]
                
                wedges, texts, autotexts = ax2.pie(
                    sizes,
                    labels=labels,
                    colors=colors_list,
                    autopct='%1.1f%%',
                    startangle=90
                )
                plt.setp(autotexts, weight='bold')
                plt.setp(texts, size=8)
                ax2.set_title(f'Unrewarded CS+ Trials (n={len(unrewarded_data)})')
            
            # Save figure
            save_path = licking_plots_dir / f'{phase_name}_reward_specific_licking_all-early.pdf'
            plt.savefig(save_path, bbox_inches='tight', dpi=300)
            plt.show()
        
        # Print summary statistics
        print(f"\nSummary for {phase_name} phase:")
        for cue_type in ['CS+', 'CS-']:
            data = all_licking_data[cue_type]
            if data:
                high_count = sum(rate > THRESHOLD for rate in data)
                low_count = len(data) - high_count
                high_percent = (high_count / len(data)) * 100
                low_percent = (low_count / len(data)) * 100
                
                print(f"\n{cue_type} (Total trials: {len(data)}):")
                print(f"High licking (> {THRESHOLD} Hz): {high_percent:.1f}% ({high_count} trials)")
                print(f"Low licking (≤ {THRESHOLD} Hz): {low_percent:.1f}% ({low_count} trials)")
        
        if phase_name == 'SucOmit':
            print("\nCS+ Reward-specific breakdown:")
            rewarded_data = all_licking_data['CS+_rewarded']
            unrewarded_data = all_licking_data['CS+_unrewarded']
            
            if rewarded_data:
                high_count = sum(rate > THRESHOLD for rate in rewarded_data)
                high_percent = (high_count / len(rewarded_data)) * 100
                print(f"Rewarded trials (n={len(rewarded_data)}): {high_percent:.1f}% high licking")
            
            if unrewarded_data:
                high_count = sum(rate > THRESHOLD for rate in unrewarded_data)
                high_percent = (high_count / len(unrewarded_data)) * 100
                print(f"Unrewarded trials (n={len(unrewarded_data)}): {high_percent:.1f}% high licking")

def perform_licking_chi_square_analysis(data_dict, save_dir=None):
    """
    Perform chi-square analysis on high vs. low licking responses across different experimental phases.
    
    Parameters:
    -----------
    data_dict : dict
        The hierarchical data dictionary from the original analysis
    save_dir : Path or str, optional
        Directory to save results
    """
    THRESHOLD = 1.5  # Hz threshold for high/low licking classification
    
    if save_dir:
        save_dir = Path(save_dir)
        stats_dir = save_dir / 'licking_chi_square_stats'
        stats_dir.mkdir(exist_ok=True)
    
    # Define the phase comparisons to analyze
    phase_comparisons = [
        ['Early', 'Late'],
        ['Late', 'SucOmit'],
        ['Rev-Early', 'Rev-Late']
    ]
    
    # Collect aggregated licking data for each phase and cue type
    phase_licking_counts = {}
    
    for phase_name, phase_data in data_dict.items():
        if not phase_data:  # Skip empty phases
            continue
            
        # Initialize counts for this phase
        phase_licking_counts[phase_name] = {
            'CS+': {'high': 0, 'low': 0},
            'CS-': {'high': 0, 'low': 0},
            'Combined': {'high': 0, 'low': 0}
        }
        
        # Aggregate licking responses across all sessions in this phase
        for session_id, session_data in phase_data.items():
            # Process CS+ trials
            for rate in session_data['CS+']['cue_delta_rates']:
                if rate > THRESHOLD:
                    phase_licking_counts[phase_name]['CS+']['high'] += 1
                    phase_licking_counts[phase_name]['Combined']['high'] += 1
                else:
                    phase_licking_counts[phase_name]['CS+']['low'] += 1
                    phase_licking_counts[phase_name]['Combined']['low'] += 1
            
            # Process CS- trials
            for rate in session_data['CS-']['cue_delta_rates']:
                if rate > THRESHOLD:
                    phase_licking_counts[phase_name]['CS-']['high'] += 1
                    phase_licking_counts[phase_name]['Combined']['high'] += 1
                else:
                    phase_licking_counts[phase_name]['CS-']['low'] += 1
                    phase_licking_counts[phase_name]['Combined']['low'] += 1
    
    # Print summary of collected data
    print("\nSummary of Licking Counts by Phase and Cue Type:")
    for phase, cue_data in phase_licking_counts.items():
        print(f"\n{phase} Phase:")
        for cue, lick_counts in cue_data.items():
            total = sum(lick_counts.values())
            print(f"  {cue} (Total: {total}):")
            for lick_type, count in lick_counts.items():
                percentage = (count / total * 100) if total > 0 else 0
                print(f"    {lick_type} licking: {count} ({percentage:.1f}%)")
    
    # Perform chi-square tests for each defined comparison
    print("\n\n====== Chi-Square Test Results for Licking Responses ======")
    
    # Store results for later visualization
    chi_results = []
    
    for phases in phase_comparisons:
        # Filter out phases that don't exist in our data
        phases = [p for p in phases if p in phase_licking_counts]
        
        if len(phases) < 2:
            print(f"Skipping comparison {phases} - insufficient data")
            continue
        
        print(f"\nComparing phases: {' vs '.join(phases)}")
        
        # Define which cue types to analyze
        cue_types_to_analyze = ['CS+', 'CS-', 'Combined']
        
        # Analyze each cue type
        for cue_type in cue_types_to_analyze:
            print(f"\n{cue_type} Analysis:")
            
            # Create contingency table: rows=phases, columns=high/low licking
            contingency_data = []
            for phase in phases:
                phase_counts = [
                    phase_licking_counts[phase][cue_type]['high'],
                    phase_licking_counts[phase][cue_type]['low']
                ]
                contingency_data.append(phase_counts)
            
            contingency_table = np.array(contingency_data)
            
            # Display contingency table
            print(f"  Contingency table (rows=phases, columns=high/low licking):")
            table_df = pd.DataFrame(contingency_data, 
                                   index=phases, 
                                   columns=['High Licking', 'Low Licking'])
            print(table_df)
            
            # Verify we have enough data for chi-square
            if np.any(contingency_table.sum(axis=0) == 0):
                print("  Error: One or more licking categories have zero counts across all phases")
                continue
            
            # Run chi-square test
            try:
                chi2, p, dof, expected = stats.chi2_contingency(contingency_table)
                
                # Check if expected frequencies are adequate
                min_expected = np.min(expected)
                if min_expected < 5:
                    warnings.warn(f"Some expected frequencies are below 5 (min={min_expected:.2f}), chi-square may not be reliable")
                    print(f"  Warning: Some expected frequencies are below 5 (min={min_expected:.2f}), chi-square may not be reliable")
            
            except Exception as e:
                print(f"  Error performing chi-square test: {str(e)}")
                continue
            
            # Display test results
            print(f"  Chi-square value: {chi2:.2f}")
            print(f"  Degrees of freedom: {dof}")
            print(f"  p-value: {p:.4f}")
            print(f"  Significant: {'Yes' if p < 0.05 else 'No'}")
            
            # If significant, calculate standardized residuals
            if p < 0.05:
                print("  Post-hoc analysis (standardized residuals):")
                row_totals = contingency_table.sum(axis=1)
                col_totals = contingency_table.sum(axis=0)
                grand_total = contingency_table.sum()
                
                # Calculate expected values
                expected_table = np.outer(row_totals, col_totals) / grand_total
                
                # Calculate standardized residuals
                standardized_residuals = (contingency_table - expected_table) / np.sqrt(expected_table)
                
                # Create DataFrame for nice output
                resid_df = pd.DataFrame(standardized_residuals, 
                                      index=phases, 
                                      columns=['High Licking', 'Low Licking'])
                print(resid_df)
                print("  Note: Absolute values > 1.96 indicate significant contribution (p < 0.05)")
                print("        Absolute values > 2.58 indicate significant contribution (p < 0.01)")
            
            # Save results for visualization
            chi_results.append({
                'comparison': ' vs '.join(phases),
                'cue_type': cue_type,
                'chi2': chi2,
                'p': p,
                'dof': dof,
                'contingency_table': table_df,
                'phases': phases
            })
    
    # Create visualizations
    print("\nCreating visualizations...")
    
    # Grouped bar chart of licking proportions by phase
    for comparison_group in phase_comparisons:
        valid_phases = [p for p in comparison_group if p in phase_licking_counts]
        if len(valid_phases) < 2:
            continue
            
        # Define which cue types to visualize
        cue_types_to_visualize = ['CS+', 'CS-', 'Combined']
            
        for cue_type in cue_types_to_visualize:
            fig, ax = plt.subplots(figsize=(12, 8))
            
            # Prepare data for plotting
            plot_data = []
            for phase in valid_phases:
                high_count = phase_licking_counts[phase][cue_type]['high']
                low_count = phase_licking_counts[phase][cue_type]['low']
                total = high_count + low_count
                
                if total == 0:
                    continue
                    
                # Add high licking data
                plot_data.append({
                    'Phase': phase,
                    'Licking': 'High',
                    'Percentage': (high_count / total) * 100,
                    'Count': high_count
                })
                
                # Add low licking data
                plot_data.append({
                    'Phase': phase,
                    'Licking': 'Low',
                    'Percentage': (low_count / total) * 100,
                    'Count': low_count
                })
            
            # Convert to DataFrame for easier plotting
            plot_df = pd.DataFrame(plot_data)
            
            # Reshape data for stacked bar chart - group by Phase
            pivoted_data = plot_df.pivot(index='Phase', columns='Licking', values='Percentage')
            pivoted_counts = plot_df.pivot(index='Phase', columns='Licking', values='Count')
            
            # Define colors based on cue type
            if cue_type == 'CS+':
                colors = ['plum', 'darkviolet']  # Low, High
            elif cue_type == 'CS-':
                colors = ['lightgrey', 'dimgrey']  # Low, High
            else:  # Combined
                colors = ['lightblue', 'darkblue']  # Low, High
            
            # Create stacked bar chart
            pivoted_data[['Low', 'High']].plot(kind='bar', stacked=True, 
                                               color=colors, ax=ax, width=0.7)
            
            # Add count labels on bar segments
            y_offset = 0
            for phase in pivoted_data.index:
                y_offset = 0
                for lick_type in ['Low', 'High']:
                    height = pivoted_data.loc[phase, lick_type]
                    count = pivoted_counts.loc[phase, lick_type]
                    
                    # Position in middle of segment
                    label_y = y_offset + (height / 2)
                    
                    # Phase index in the plot
                    phase_idx = pivoted_data.index.get_loc(phase)
                    
                    # Add text
                    ax.text(phase_idx, label_y, f'{count}', 
                            ha='center', va='center', fontsize=9, 
                            color='black' if lick_type == 'Low' else 'white',
                            fontweight='bold')
                    
                    y_offset += height 
            
            # Set title and labels
            ax.set_title(f'Licking Responses: {cue_type} Across {" vs ".join(valid_phases)}')
            ax.set_ylabel('Percentage (%)')
            ax.set_ylim(0, 100)
            
            # Add chi-square results as text
            # Find matching chi-square result
            for result in chi_results:
                if result['cue_type'] == cue_type and all(p in result['comparison'] for p in valid_phases):
                    plt.figtext(0.5, 0.01, 
                               f"Chi-square: {result['chi2']:.2f}, p-value: {result['p']:.4f}, " + 
                               f"Significant: {'Yes' if result['p'] < 0.05 else 'No'}", 
                               ha="center", fontsize=10)
                    break
            
            plt.tight_layout()
            
            if save_dir:
                plt.savefig(stats_dir / f"licking_{cue_type}_{'_vs_'.join(valid_phases)}_all-early.png", 
                           dpi=300, bbox_inches='tight')
                plt.savefig(stats_dir / f"licking_{cue_type}_{'_vs_'.join(valid_phases)}_all-early.pdf", 
                           bbox_inches='tight')
            plt.show()
    
    print("\nLicking chi-square analysis complete. Results saved to:", stats_dir if save_dir else "Not saved")
    return chi_results

# Run the analysis as a continuation of the original script
print("\n\n======= Running Licking Response Analysis =======")

# Define base directory (same as in original script)
basedir = DATA_ROOT
results_dir = Path(basedir) / 'Results'

# Run the licking analysis visualization
visualize_licking_response(data_dict, results_dir)

# Run the chi-square analysis
licking_chi_results = perform_licking_chi_square_analysis(data_dict, results_dir)

print("\nLicking analysis complete!")

In [ ]:
###this portion of the code is looking at the signal detection theory metrics like sensitivity, response bias and rates of response types
        ###it is using the delta lick rates and the threshold of 1.5 as a cut off for high/low licking

def calculate_sdt_metrics(data_dict, results_dir):
    """
    Calculate SDT-like metrics for each session using the response classifications.
    Handles both initial learning and reversal phases correctly.
    """
    import numpy as np
    from scipy.stats import norm
    from pathlib import Path
    import pandas as pd

    def calculate_rates(responses):
        """Calculate proportions of each response type"""
        total = len(responses)
        if total == 0:
            return {}
        
        counts = {}
        for response in responses:
            counts[response] = counts.get(response, 0) + 1
            
        return {resp: count/total for resp, count in counts.items()}

    def calculate_d_prime_beta(hit_rate, fa_rate):
        """Calculate d' and β using correct_response_rate and incorrect_response_rate"""
        # Add small epsilon to avoid infinity in norm.ppf
        epsilon = 0.001
        hit_rate = min(max(hit_rate, epsilon), 1-epsilon)
        fa_rate = min(max(fa_rate, epsilon), 1-epsilon)
        
        z_hit = norm.ppf(hit_rate)
        z_fa = norm.ppf(fa_rate)
        
        d_prime = z_hit - z_fa
        beta = np.exp((z_fa**2 - z_hit**2)/2)
        
        return d_prime, beta

    metrics_data = []
    
    for phase in data_dict.keys():
        print(f"\nProcessing {phase} Phase")
        phase_data = data_dict[phase]
        is_reversal = phase.startswith('Rev-')
        
        for session_id, session_data in phase_data.items():
            # Get response rates for both cues
            cs_plus_rates = calculate_rates(session_data['CS+']['responses'])
            cs_minus_rates = calculate_rates(session_data['CS-']['responses'])
            
            if not is_reversal:
                # Initial learning phases
                correct_response_rate = cs_plus_rates.get('correct response', 0)
                incorrect_withholding_rate = cs_plus_rates.get('incorrect withholding', 0)
                correct_withholding_rate = cs_minus_rates.get('correct withholding', 0)
                incorrect_response_rate = cs_minus_rates.get('incorrect response', 0)
            else:
                # Reversal phases - switch the interpretation
                correct_response_rate = cs_minus_rates.get('correct response', 0)
                incorrect_withholding_rate = cs_minus_rates.get('incorrect withholding', 0)
                correct_withholding_rate = cs_plus_rates.get('correct withholding', 0)
                incorrect_response_rate = cs_plus_rates.get('incorrect response', 0)
            
            # Calculate d' and β using the appropriate rates
            d_prime, beta = calculate_d_prime_beta(correct_response_rate, incorrect_response_rate)
            
            # Store all metrics
            metrics_data.append({
                'Phase': phase,
                'Session': session_id,
                'CS+_correct_response_rate': correct_response_rate,
                'CS+_incorrect_withholding_rate': incorrect_withholding_rate,
                'CS-_correct_withholding_rate': correct_withholding_rate,
                'CS-_incorrect_response_rate': incorrect_response_rate,
                'd_prime': d_prime,
                'beta': beta
            })
            
            # Update the data dictionary with SDT metrics
            session_data['sdt_metrics'] = {
                'CS+': {
                    'correct_response_rate': correct_response_rate,
                    'incorrect_withholding_rate': incorrect_withholding_rate
                },
                'CS-': {
                    'correct_withholding_rate': correct_withholding_rate,
                    'incorrect_response_rate': incorrect_response_rate
                },
                'overall': {
                    'd_prime': d_prime,
                    'beta': beta
                }
            }
    
    # Create DataFrame and save metrics
    df = pd.DataFrame(metrics_data)
    metrics_path = Path(results_dir) / 'sdt_metrics_1-5hz.csv'
    df.to_csv(metrics_path, index=False)
    print(f"\nSDT metrics saved to: {metrics_path}")
    
    return df, data_dict  # Return both the metrics DataFrame and updated dictionary

def analyze_sdt_statistics(sdt_df, results_dir):
    """
    Perform statistical analyses on SDT metrics across phases.
    Uses independent t-tests for Early vs Late, Late vs SucOmit, and Rev-Early vs Rev-Late.
    CueOmit is excluded from the analyses.
    Includes descriptive statistics (mean and SEM) for each comparison.
    """
    from scipy import stats
    import pandas as pd
    from pathlib import Path

    # Define metrics to analyze
    metrics = ['d_prime', 'beta', 'CS+_correct_response_rate', 'CS-_correct_withholding_rate']
    
    stats_results = []
    
    # Function for independent t-test comparisons
    def run_independent_ttest(data, phase1, phase2, metric):
        group1 = data[data['Phase'] == phase1][metric]
        group2 = data[data['Phase'] == phase2][metric]
        
        if len(group1) > 1 and len(group2) > 1:  # Need at least 2 samples per group
            t_stat, p_val = stats.ttest_ind(group1, group2)
            df_val = len(group1) + len(group2) - 2  # degrees of freedom for independent t-test
            return {
                'Comparison': f'{phase1} vs {phase2}',
                'Metric': metric,
                'Test': 'Independent t-test',
                'Statistic': t_stat,
                'df': df_val,
                'p-value': p_val,
                'n1': len(group1),
                'n2': len(group2),
                'mean1': group1.mean(),
                'mean2': group2.mean(),
                'sem1': group1.sem(),
                'sem2': group2.sem()
            }
        else:
            return {
                'Comparison': f'{phase1} vs {phase2}',
                'Metric': metric,
                'Test': 'Independent t-test',
                'Statistic': 'Insufficient data',
                'df': 'NA',
                'p-value': 'NA',
                'n1': len(group1),
                'n2': len(group2),
                'mean1': group1.mean() if len(group1) > 0 else 'NA',
                'mean2': group2.mean() if len(group2) > 0 else 'NA',
                'sem1': group1.sem() if len(group1) > 1 else 'NA',
                'sem2': group2.sem() if len(group2) > 1 else 'NA'
            }

    # Run analyses for each metric
    for metric in metrics:
        print(f"\nAnalyzing {metric}...")
        
        # Early vs Late independent t-test
        stats_results.append(
            run_independent_ttest(sdt_df, 'Early', 'Late', metric)
        )
        
        # Late vs SucOmit independent t-test (replaced ANOVA)
        stats_results.append(
            run_independent_ttest(sdt_df, 'Late', 'SucOmit', metric)
        )
        
        # Rev-Early vs Rev-Late independent t-test
        stats_results.append(
            run_independent_ttest(sdt_df, 'Rev-Early', 'Rev-Late', metric)
        )
    
    # Convert results to DataFrame
    results_df = pd.DataFrame(stats_results)
    
    # Print formatted results tables
    print("\nIndependent t-test Analysis Results:")
    print("=" * 80)
    
    # Define comparisons for table formatting
    comparisons = [
        ('Early', 'Late'),
        ('Late', 'SucOmit'), 
        ('Rev-Early', 'Rev-Late')
    ]
    
    for phase1, phase2 in comparisons:
        comparison_name = f'{phase1} vs {phase2}'
        comparison_data = results_df[results_df['Comparison'] == comparison_name]
        
        if not comparison_data.empty:
            print(f"\n{comparison_name}")
            print("-" * 60)
            
            # Create formatted table
            table_data = []
            for _, row in comparison_data.iterrows():
                if row['p-value'] != 'NA':
                    table_data.append({
                        'Metric': row['Metric'],
                        f'{phase1} Mean±SEM': f"{row['mean1']:.3f}±{row['sem1']:.3f}",
                        f'{phase2} Mean±SEM': f"{row['mean2']:.3f}±{row['sem2']:.3f}",
                        'n1': int(row['n1']),
                        'n2': int(row['n2']),
                        't': f"{row['Statistic']:.3f}",
                        'df': int(row['df']),
                        'p-value': f"{row['p-value']:.3f}" if row['p-value'] >= 0.001 else f"{row['p-value']:.2e}"
                    })
                else:
                    table_data.append({
                        'Metric': row['Metric'],
                        f'{phase1} Mean±SEM': f"{row['mean1']}±{row['sem1']}" if row['mean1'] != 'NA' else 'NA',
                        f'{phase2} Mean±SEM': f"{row['mean2']}±{row['sem2']}" if row['mean2'] != 'NA' else 'NA',
                        'n1': int(row['n1']) if row['n1'] != 'NA' else 'NA',
                        'n2': int(row['n2']) if row['n2'] != 'NA' else 'NA',
                        't': row['Statistic'],
                        'df': row['df'],
                        'p-value': row['p-value']
                    })
            
            # Convert to DataFrame and display
            display_df = pd.DataFrame(table_data)
            print(display_df.to_string(index=False))
            print()
    
    # Save results to CSV
    results_path = Path(results_dir) / 'independent_ttest_results.csv'
    results_df.to_csv(results_path, index=False)
    print(f"Independent t-test results saved to: {results_path}")
    
    return results_df

def visualize_sdt_metrics(sdt_df, results_dir):
    """
    Create bar plots with SEMs and individual animal data points for SDT metrics,
    handling reversal phases separately.
    CueOmit is excluded from the visualizations.
    """
    import seaborn as sns
    import matplotlib.pyplot as plt
    import numpy as np
    from pathlib import Path
    
    # Create directory for SDT plots
    sdt_plots_dir = Path(results_dir) / 'sdt_plots'
    sdt_plots_dir.mkdir(exist_ok=True)
    
    # Set style parameters
    plt.style.use('seaborn')
    colors = sns.color_palette("deep")
    
    # Define phase comparisons
    phase_comparisons = [
        {
            'phases': ['Early', 'Late'],
            'title': 'Early vs Late Learning',
            'filename': 'early_vs_late'
        },
        {
            'phases': ['Late', 'SucOmit'],  # Removed CueOmit
            'title': 'Late Learning vs Omission Tests',
            'filename': 'late_vs_omission'
        },
        {
            'phases': ['Rev-Early', 'Rev-Late'],
            'title': 'Reversal Learning',
            'filename': 'reversal'
        }
    ]
    
    # Define metrics to plot
    metrics = [
        {
            'name': 'd_prime',
            'title': "d' (Sensitivity)",
            'filename': 'dprime'
        },
        {
            'name': 'beta',
            'title': 'β (Response Bias)',
            'filename': 'beta'
        },
        {
            'name': 'CS+_correct_response_rate',
            'title': 'CS+ Correct Response Rate',
            'filename': 'csplus_correct'
        },
        {
            'name': 'CS-_correct_withholding_rate',
            'title': 'CS- Correct Withholding Rate',
            'filename': 'csminus_correct'
        }
    ]
    
    def plot_bar_with_points(data, metric, phases, ax):
        """Helper function to create bar plot with points and SEMs"""
        # Calculate means and SEMs for each phase
        means = []
        sems = []
        for phase in phases:
            phase_data = data[data['Phase'] == phase][metric]
            means.append(phase_data.mean())
            sems.append(phase_data.sem())
        
        # Plot bars
        bar_positions = np.arange(len(phases))
        bars = ax.bar(bar_positions, means, yerr=sems, capsize=5, alpha=0.6, color='m')
        
        # Plot individual points
        for i, phase in enumerate(phases):
            phase_data = data[data['Phase'] == phase][metric]
            # Add jitter to x-positions
            x_jitter = np.random.normal(i, 0.00, size=len(phase_data))
            ax.scatter(x_jitter, phase_data, color='m', alpha=0.6, s=40)
        
        return bars
    
    # Create plots for each comparison and metric
    for comparison in phase_comparisons:
        # Filter data for current phases
        phase_data = sdt_df[sdt_df['Phase'].isin(comparison['phases'])]
        
        for metric in metrics:
            # Create figure
            fig, ax = plt.subplots(figsize=(10, 6))
            
            # Create bar plot with points
            bars = plot_bar_with_points(phase_data, metric['name'], 
                                      comparison['phases'], ax)
            
            # Customize plot
            # Add note for reversal phases if applicable
            if 'Rev' in comparison['phases'][0]:
                title = f"{comparison['title']}\n{metric['title']}\n(Contingencies Reversed)"
            else:
                title = f"{comparison['title']}\n{metric['title']}"
            
            ax.set_title(title, pad=20)
            ax.set_xlabel('Learning Phase')
            ax.set_ylabel(metric['title'])
            ax.set_xticks(range(len(comparison['phases'])))
            
            # Clean up phase labels for reversal
            xticklabels = [phase.replace('Rev-', '') for phase in comparison['phases']]
            ax.set_xticklabels(xticklabels)
            
            # Add gridlines
            ax.yaxis.grid(True, linestyle='--', alpha=0.7)
            ax.set_axisbelow(True)
            
            # Adjust layout
            plt.tight_layout()
            
            # Save plot
            save_path = sdt_plots_dir / f"{comparison['filename']}_{metric['filename']}_03-18-2025_all-early.pdf"
            plt.savefig(save_path, bbox_inches='tight', dpi=300)
            plt.show()
            
    print(f"\nSDT visualization plots saved in: {sdt_plots_dir}")

# Example usage
if __name__ == "__main__":
    # Using the same data_dict and results_dir from previous code
    print("Checking data dictionary...")
    if 'data_dict' in globals():
        print(f"Found data dictionary with {len(data_dict)} phases")
    else:
        print("Data dictionary not found. Please run the previous cell first.")
        
    print("\nCalculating SDT metrics...")
    sdt_df, data_dict = calculate_sdt_metrics(data_dict, results_dir)

    print("\nPerforming statistical analyses...")
    stats_results = analyze_sdt_statistics(sdt_df, results_dir)

    print("\nCreating SDT visualizations...")
    visualize_sdt_metrics(sdt_df, results_dir)

    print("\nAnalysis complete!")